# Salesforce to Databricks Pipeline Generator

This notebook helps you generate Databricks Asset Bundle YAML files for Salesforce ingestion pipelines.

## How it works:
1. **Input CSV with prefix + priority**: You provide a CSV with Salesforce objects grouped by business units (prefix) and priorities
2. **Pipeline Grouping**: Each unique (prefix, priority) combination creates a separate pipeline
3. **YAML Generation**: Generates Databricks Asset Bundle configuration with scheduled jobs
4. **Deploy**: Use Databricks CLI to deploy the pipelines

## Pipeline Naming Convention:
- Pipeline name: `{prefix}_{priority}`
- Example: `business_unit1_01`, `sales_team_02`, etc.

dbutils.library.restartPython()

# Install required dependencies
%pip install pyyaml pandas

# Step 1: Prepare Input CSV with Prefix and Priority

Your input CSV should contain:
- **prefix**: Business unit or logical grouping (e.g., "business_unit1", "sales_team", "finance")
- **priority**: Priority level within that prefix (e.g., "01", "02", "03")

Each unique combination of (prefix, priority) will create a separate pipeline named: `{prefix}_{priority}`

Example: `business_unit1_01`, `business_unit2_02`, etc.

In [0]:
# View your input configuration CSV
# Make sure it has the required columns: prefix, priority, source_table_name, target_catalog, target_schema, target_table_name, connection_name, schedule

%sql
SELECT * FROM read_files(
  '/Volumes/your_catalog/your_schema/your_volume/input_config.csv',
  format => 'csv',
  header => true
)

In [0]:
%pip install pyyaml

In [0]:
# Step 2: Generate Pipeline Configuration (Grouping by prefix + priority)

import sys
sys.path.append('..')  # Add parent directory to path to import from load_balancing

from load_balancing.generate_pipeline_config import generate_pipeline_config
import pandas as pd

# Configuration
INPUT_CSV = "/Volumes/your_catalog/your_schema/your_volume/input_config.csv"
OUTPUT_CONFIG = "/Workspace/Users/your_email@domain.com/sfdc_pipeline_config.csv"
DEFAULT_CONNECTION = "sfdc_connection"
DEFAULT_SCHEDULE = "*/15 * * * *"

# Read input CSV from volume
input_df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(input_df)} Salesforce objects")

# Generate pipeline configuration using prefix + priority
config_df = generate_pipeline_config(
    df=input_df,
    default_connection_name=DEFAULT_CONNECTION,
    default_schedule=DEFAULT_SCHEDULE
)

# Save intermediate configuration
config_df.to_csv(OUTPUT_CONFIG, index=False)
print(f"\n✓ Pipeline configuration saved to: {OUTPUT_CONFIG}")

In [0]:
# Step 3: Generate Databricks Asset Bundle YAML

from deployment.generate_dab_yaml import generate_salesforce_yaml
import pandas as pd

# Configuration
PIPELINE_CONFIG_CSV = "/Workspace/Users/your_email@domain.com/sfdc_pipeline_config.csv"
OUTPUT_YAML_DIR = "/Workspace/Users/your_email@domain.com/sfdc_deployment/resources"
CONNECTION_NAME = "sfdc_connection"  # Your Salesforce connection name in Databricks

# Read the pipeline configuration
config_df = pd.read_csv(PIPELINE_CONFIG_CSV)

# Generate the YAML file
output_yaml_path = f"{OUTPUT_YAML_DIR}/sfdc_pipeline.yml"
generate_salesforce_yaml(
    df=config_df,
    connection_name=CONNECTION_NAME,
    output_path=output_yaml_path
)

print(f"\n✓ YAML generated successfully!")
print(f"  Output: {output_yaml_path}")

# Alternative: Run Complete Pipeline Generation in One Step

import sys
sys.path.append('..')

from run_pipeline_generation import run_complete_pipeline_generation

# Configuration
INPUT_CSV = "/Volumes/your_catalog/your_schema/your_volume/input_config.csv"
OUTPUT_YAML = "/Workspace/Users/your_email@domain.com/sfdc_deployment/resources/sfdc_pipeline.yml"
OUTPUT_CONFIG = "/Workspace/Users/your_email@domain.com/sfdc_pipeline_config.csv"
CONNECTION_NAME = "sfdc_connection"
SCHEDULE = "*/15 * * * *"

# Run complete pipeline generation
result_df = run_complete_pipeline_generation(
    input_csv=INPUT_CSV,
    default_connection_name=CONNECTION_NAME,
    default_schedule=SCHEDULE,
    output_yaml=OUTPUT_YAML,
    output_config=OUTPUT_CONFIG
)

print("\n✓ Pipeline generation complete!")
print(f"  Total pipelines: {result_df['pipeline_group'].nunique()}")
print(f"  Total tables: {len(result_df)}")

In [0]:
# Deploy using Databricks Asset Bundles CLI

# Note: Run these commands in a terminal or notebook cell with ! prefix

# 1. Install dependencies
!pip install pyyaml pandas

# 2. Navigate to your deployment directory
cd /Workspace/Users/your_email@domain.com/sfdc_deployment

# 3. Validate the bundle
!databricks bundle validate -t dev

# 4. Deploy to Databricks
!databricks bundle deploy -t dev

# 5. (Optional) View deployed resources
!databricks pipelines list
!databricks jobs list

# View Generated Pipeline Configuration

import pandas as pd

# Read the generated configuration
config_df = pd.read_csv("/Workspace/Users/your_email@domain.com/sfdc_pipeline_config.csv")

# Show summary
print(f"Total tables: {len(config_df)}")
print(f"Total pipelines: {config_df['pipeline_group'].nunique()}")
print(f"\nPipelines by prefix:")
print(config_df.groupby('prefix')['pipeline_group'].nunique())

# Show detailed breakdown
print(f"\nDetailed pipeline breakdown:")
for pipeline in sorted(config_df['pipeline_group'].unique()):
    tables = config_df[config_df['pipeline_group'] == pipeline]['source_table_name'].tolist()
    schedule = config_df[config_df['pipeline_group'] == pipeline]['schedule'].iloc[0]
    print(f"\n{pipeline} (Schedule: {schedule}):")
    for table in tables:
        print(f"  - {table}")

# Display the dataframe
display(config_df)

# Start the deployed pipelines

# Get list of deployed pipelines
!databricks pipelines list

# Start a specific pipeline (replace with your pipeline ID)
# !databricks pipelines start-update <pipeline-id>

# Or start all pipelines from a specific prefix (e.g., business_unit1)
# Use the Databricks SDK
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# List all pipelines with "SFDC Ingestion" in the name
pipelines = [p for p in w.pipelines.list_pipelines() if "SFDC Ingestion" in p.name]

print(f"Found {len(pipelines)} SFDC pipelines:")
for p in pipelines:
    print(f"  - {p.name} (ID: {p.pipeline_id})")
    
# Uncomment to start a specific pipeline
# w.pipelines.start_update(pipeline_id="your-pipeline-id")